# Modeling Strategy Implementation

### Covered Workflow
- Load training and test data
- Load Stage 3 feature-set scores from `outputs/stage3_feature_set_scores.csv`
- Compare model families across the precomputed feature subsets
- Perform per-model hyperparameter optimization (HPO) for all candidate model families and persist results to `outputs/hpo/`
- Tune the final business score using out-of-fold probabilities
- Fit the final model and export ranked predictions

### Inputs and Outputs
- Input ranking artifact: `outputs/stage3_feature_set_scores.csv`
- HPO artifacts: `outputs/hpo/{model_name}_hpo.json`, `outputs/hpo/hpo_summary.json`
- Optimal threshold (from baseline): `outputs/optimal_threshold.json`
- Output predictions: `outputs/model_predictions.csv`

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| candidate_feature_sizes | (1, 3, 5, 8, 10, 15, 20, all) | Top-k feature subsets evaluated in modeling |
| modeling_cv_folds | 5 | Cross-validation folds used for model comparison |
| max_test_targets | 1000 | Maximum ranked test samples to retain |
| random_state | 42 | Random seed |
| include_xgboost | False | Optional XGBoost factory when the dependency is available |

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from tqdm.notebook import tqdm

from cost_effective.dataset import (
    business_scorer_no_var_penalty,
    custom_scorer,
    f1_scorer_wrapper,
    get_classifier,
)
from cost_effective.models import (
    build_f1_curve,
    build_model_factories,
    build_profit_curve,
    compare_models_on_feature_sets,
    compute_oof_probabilities,
    fit_final_model_and_predict,
)

PROJECT_ROOT: Path = Path().resolve().parent
DATA_PATH: Path = PROJECT_ROOT / "data"
OUTPUTS_PATH: Path = PROJECT_ROOT / "outputs"


LOGISTIC_PARAM_GRID: dict[str, list] = {
    "logisticregression__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "logisticregression__penalty": ["l1", "l2"],
}

LGB_PARAM_DIST: dict[str, list] = {
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 4, 6, 8, -1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

XGB_PARAM_DIST: dict[str, list] = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 6, 8],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

In [3]:
try:
    with (OUTPUTS_PATH / "optimal_threshold.json").open("r") as f:
        threshold_config = json.load(f)
    optimal_threshold = threshold_config["optimal_threshold"]
    print(f"✓ Optimal threshold loaded from baseline: {optimal_threshold:.3f}")
except FileNotFoundError:
    print("⚠ Optimal threshold file not found; defaulting to 0.5")
    optimal_threshold = 0.5

✓ Optimal threshold loaded from baseline: 0.290


In [4]:
X_train = pd.read_csv(
    DATA_PATH / "x_train.txt",
    sep=r"\s+",
    header=None,
    skiprows=1,
    low_memory=False,
).apply(pd.to_numeric, errors="coerce")
X_train.fillna(0.0)
X_train.columns = [f"var_{i}" for i in range(X_train.shape[1])]

y_train = pd.read_csv(DATA_PATH / "y_train.txt", header=None, skiprows=1).iloc[:, 0].astype(int)

X_test = pd.read_csv(
    DATA_PATH / "x_test.txt",
    sep=r"\s+",
    header=None,
    skiprows=1,
    low_memory=False,
).apply(pd.to_numeric, errors="coerce")
X_test.fillna(0.0)
X_test.columns = [f"var_{i}" for i in range(X_test.shape[1])]

stage3_scores = pd.read_csv(OUTPUTS_PATH / "stage3_feature_set_scores.csv")
stage3_scores["features"] = stage3_scores["features"].apply(ast.literal_eval)

stage2_row = stage3_scores.loc[stage3_scores["feature_count"].idxmax()]
stage2_features = stage2_row["features"]
feature_set_candidates = {
    row.feature_set_name: row.features for row in stage3_scores.itertuples(index=False)
}
X_stage2 = X_train[stage2_features]

print(f"✓ Loaded training data: {X_train.shape}")
print(f"✓ Loaded test data: {X_test.shape}")
print(f"✓ Loaded labels: {len(y_train)} rows")
print(f"✓ Loaded Stage 3 score rows: {len(stage3_scores)}")
print(f"✓ Loaded Stage 2 feature set: {len(stage2_features)} features")

✓ Loaded training data: (5000, 500)
✓ Loaded test data: (5000, 500)
✓ Loaded labels: 5000 rows
✓ Loaded Stage 3 score rows: 8
✓ Loaded Stage 2 feature set: 26 features


In [ ]:
base_factories = build_model_factories(y_train)

kw = {
    "scoring": {
        "business": custom_scorer,
        "f1": f1_scorer_wrapper,
        "business_no_var": business_scorer_no_var_penalty,
        "roc_auc": "roc_auc",
    },
    "verbose": 2,
    "n_jobs": -1,
    "cv": 5,
    "refit": "business",
}
n_iter = 20

hpo_results = {}

# tuned_factories will hold clones of best estimators
tuned_factories = {}

hpo_out_dir = OUTPUTS_PATH / "hpo"
hpo_out_dir.mkdir(parents=True, exist_ok=True)

pbar = tqdm(base_factories.items(), desc="Hyperparameter Optimization", unit="model")
for model_name, factory in pbar:
    pbar.set_postfix_str(f"Tuning {model_name}")

    if model_name == "logistic_regression":
        base_est = factory()
        # Grid over pipeline parameters
        gs = GridSearchCV(base_est, LOGISTIC_PARAM_GRID, **kw)
        gs.fit(X_stage2, y_train)
        best = gs.best_estimator_
        best_params = gs.best_params_

    elif model_name == "lightgbm":
        base_est = get_classifier(y_train.to_numpy())
        rs = RandomizedSearchCV(base_est, LGB_PARAM_DIST, n_iter=n_iter, random_state=42, **kw)
        rs.fit(X_stage2, y_train)
        best = rs.best_estimator_
        best_params = rs.best_params_

    elif model_name == "xgboost":
        base_est = factory()
        rs = RandomizedSearchCV(base_est, XGB_PARAM_DIST, n_iter=n_iter, random_state=42, **kw)
        rs.fit(X_stage2, y_train)
        best = rs.best_estimator_
        best_params = rs.best_params_

    else:
        raise ValueError(f"HPO not configured for model: {model_name}")

    # Save HPO results and create a cloning factory for the best estimator
    hpo_results[model_name] = {
        "best_params": best_params,
        "score": float(custom_scorer(best, X_stage2.to_numpy(), y_train.to_numpy())),
    }
    # Persist per-model HPO summary
    with (hpo_out_dir / f"{model_name}_hpo.json").open("w") as fh:
        json.dump(hpo_results[model_name], fh, indent=2)

    # Factory that clones the fitted estimator
    tuned_factories[model_name] = (lambda est=best: lambda *_: clone(est))()

# Save aggregate HPO results
with (hpo_out_dir / "hpo_summary.json").open("w") as fh:
    json.dump(hpo_results, fh, indent=2)

# Run model comparison using tuned factories
model_comparison = compare_models_on_feature_sets(
    X_stage2,
    y_train,
    feature_set_candidates,
    estimator_factories=tuned_factories,
    cv=5,
    threshold=optimal_threshold,
)


# Attach best hyperparameters to the comparison table where available
def _params_for(model_name):
    return hpo_results.get(model_name, {}).get("best_params", {})


model_comparison["best_hyperparams"] = model_comparison["model_name"].apply(_params_for)

# Pick the best business row (first row after sorting) if present
best_business_row = model_comparison.iloc[0] if len(model_comparison) else None

# Safely pick the best ROC AUC row — handle missing or all-NaN cases
if (
    "roc_auc_score" in model_comparison.columns
    and model_comparison["roc_auc_score"].dropna().size > 0
):
    best_model_idx = model_comparison["roc_auc_score"].idxmax()
    best_model_row = model_comparison.loc[best_model_idx]
else:
    best_model_row = None

best_model_name = str(best_model_row["model_name"]) if best_model_row is not None else None
best_feature_set_name = (
    str(best_model_row["feature_set_name"]) if best_model_row is not None else None
)
best_features = feature_set_candidates.get(best_feature_set_name, [])

if best_business_row is not None:
    print(
        f"Best business model: {best_business_row['model_name']} / "
        f"{best_business_row['feature_set_name']}"
    )
else:
    print("No business-best row available")

if best_model_row is not None:
    print(
        f"Best ROC AUC model: {best_model_name} / "
        f"{best_feature_set_name} ({len(best_features)} features)"
    )
else:
    print("No ROC AUC model available")

model_comparison

Hyperparameter Optimization:   0%|          | 0/3 [00:00<?, ?model/s]

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=400, num_leaves=127, subsample=0.8; total time=   2.4s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=400, num_leaves=127, subsample=0.8; total time=   2.5s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=400, num_leaves=127, subsample=0.8; total time=   2.5s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=400, num_leaves=127, subsample=0.8; total time=   2.5s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=400, num_leaves=127, subsample=0.8; total time=   2.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, num_leaves=15, subsample=1.0; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, num_leaves=15, subsample=1.0; total time=   2.0s
[CV] END colsample_by

,model_name,feature_set_name,feature_count,cv_score_mean,cv_score_std,f1_score,roc_auc_score,business_score_no_var_penalty,best_hyperparams
0,lightgbm,top_01,1,2264.0,7.348469,0.664530,0.523158,2464.0,"{'subsample': 0.6, 'num_leaves': 31, 'n_estima..."
1,logistic_regression,top_01,1,2264.0,7.348469,0.664530,0.512797,2464.0,"{'logisticregression__C': 0.1, 'logisticregres..."
2,xgboost,top_01,1,2264.0,7.348469,0.664530,0.508903,2464.0,"{'subsample': 0.6, 'n_estimators': 200, 'max_d..."
3,lightgbm,top_03,3,1864.0,7.348469,0.664530,0.614642,2464.0,"{'subsample': 0.6, 'num_leaves': 31, 'n_estima..."
4,logistic_regression,top_03,3,1864.0,7.348469,0.664530,0.570100,2464.0,"{'logisticregression__C': 0.1, 'logisticregres..."
5,xgboost,top_03,3,1864.0,7.348469,0.664530,0.592616,2464.0,"{'subsample': 0.6, 'n_estimators': 200, 'max_d..."
6,xgboost,top_05,5,1473.0,7.483315,0.665328,0.627896,2473.0,"{'subsample': 0.6, 'n_estimators': 200, 'max_d..."
7,lightgbm,top_05,5,1467.0,24.413111,0.664783,0.626709,2467.0,"{'subsample': 0.6, 'num_leaves': 31, 'n_estima..."
8,logistic_regression,top_05,5,1463.0,28.390139,0.664429,0.587832,2463.0,"{'logisticregression__C': 0.1, 'logisticregres..."
9,lightgbm,top_08,8,876.0,23.108440,0.665592,0.627274,2476.0,"{'subsample': 0.6, 'num_leaves': 31, 'n_estima..."


In [ ]:
roc_model_comparison = compare_models_on_feature_sets(
    X_stage2,
    y_train,
    feature_set_candidates,
    estimator_factories=tuned_factories,
    cv=5,
    threshold=None,
)

roc_model_comparison["best_hyperparams"] = roc_model_comparison["model_name"].apply(_params_for)

best_f1_model_row = roc_model_comparison.loc[roc_model_comparison["f1_score"].idxmax()]
best_model_name = str(best_f1_model_row["model_name"])
best_feature_set_name = str(best_f1_model_row["feature_set_name"])
best_features = feature_set_candidates[best_feature_set_name]

print(f"Best F1 model: {best_model_name} / {best_feature_set_name} ({len(best_features)} features)")
print(
    roc_model_comparison[
        ["model_name", "feature_set_name", "cv_score_mean", "f1_score", "roc_auc_score"]
    ]
    .sort_values("f1_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

Best F1 model: xgboost / top_10 (10 features)
         model_name feature_set_name  cv_score_mean  f1_score  roc_auc_score
            xgboost           top_10          460.0  0.579285       0.642099
            xgboost           top_15         -535.0  0.575893       0.637612
logistic_regression           top_26        -2727.0  0.573975       0.608609
            xgboost           top_26        -2721.0  0.573750       0.637254
logistic_regression           top_15         -528.0  0.571713       0.608138
logistic_regression           top_20        -1539.0  0.570988       0.607455
logistic_regression           top_10          471.0  0.569512       0.607527
            xgboost           top_20        -1528.0  0.568992       0.639930
           lightgbm           top_10          478.0  0.565284       0.639119
            xgboost           top_08          868.0  0.564014       0.629343


In [ ]:
best_business_row = model_comparison.loc[model_comparison["cv_score_mean"].idxmax()]
print(f"Best business score (CV mean): {best_business_row['cv_score_mean']:.2f}")
print(f"Best feature count: {int(best_business_row['feature_count'])}")
print(f"Thresholded F1: {best_business_row['f1_score']:.4f}")
print(
    "Business score without feature penalty: "
    f"{best_business_row['business_score_no_var_penalty']:.2f}"
)

Best business score (CV mean): 2264.00
Best feature count: 1
Thresholded F1: 0.6645
Business score without feature penalty: 2464.00


In [ ]:
oof_probabilities = compute_oof_probabilities(
    X_stage2[best_features],
    y_train,
    estimator_factory=tuned_factories.get(best_model_name, base_factories.get(best_model_name)),
    cv=5,
)
business_curve = build_profit_curve(
    y_train,
    oof_probabilities,
    feature_count=len(best_features),
    max_targets=10000,
)
f1_curve = build_f1_curve(
    y_train,
    oof_probabilities,
    max_targets=10000,
)

business_best_row = business_curve.curve.loc[business_curve.curve["score"].idxmax()]
f1_best_row = f1_curve.curve.loc[f1_curve.curve["f1"].idxmax()]
selected_threshold = f1_curve.best_threshold
y_pred_oof = (oof_probabilities >= selected_threshold).astype(int)
confusion_df = pd.DataFrame(
    confusion_matrix(y_train, y_pred_oof, labels=[0, 1]),
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)

print(
    f"Business-optimal cutoff: k={business_curve.best_k}, "
    f"threshold={business_curve.best_threshold:.4f}, "
    f"score={business_curve.best_score:.2f}"
)
print(
    f"F1-optimal cutoff: k={f1_curve.best_k}, threshold={f1_curve.best_threshold:.4f}, "
    f"f1={f1_curve.best_f1:.4f}, roc_auc={f1_curve.roc_auc:.4f}, "
    f"avg_precision={f1_curve.average_precision:.4f}"
)
print(f"Business curve TP = {int(business_best_row['tp'])}")
print(f"Business curve FP = {int(business_best_row['fp'])}")
print(f"F1 curve TP = {int(f1_best_row['tp'])}")
print(f"F1 curve FP = {int(f1_best_row['fp'])}")
print("Confusion matrix (OOF, F1 threshold):")
print(confusion_df.to_string())

f1_curve.curve.head(10)

Business-optimal cutoff: k=4739, threshold=0.3581, score=10425.00
F1-optimal cutoff: k=4739, threshold=0.3581, f1=0.6664, roc_auc=0.6390, avg_precision=0.6268
Business curve TP = 2408
Business curve FP = 2331
F1 curve TP = 2408
F1 curve FP = 2331
Confusion matrix (OOF, F1 threshold):
          Predicted 0  Predicted 1
Actual 0          181         2331
Actual 1           80         2408


,k,tp,fp,fn,tn,precision,recall,f1,accuracy,specificity,threshold
0,1,0,1,2488,2511,0.000000,0.000000,0.000000,0.5022,0.999602,0.819607
1,2,0,2,2488,2510,0.000000,0.000000,0.000000,0.5020,0.999204,0.807242
2,3,1,2,2487,2510,0.333333,0.000402,0.000803,0.5022,0.999204,0.806082
3,4,1,3,2487,2509,0.250000,0.000402,0.000803,0.5020,0.998806,0.804635
4,5,2,3,2486,2509,0.400000,0.000804,0.001604,0.5022,0.998806,0.801248
5,6,3,3,2485,2509,0.500000,0.001206,0.002406,0.5024,0.998806,0.795354
6,7,4,3,2484,2509,0.571429,0.001608,0.003206,0.5026,0.998806,0.794406
7,8,5,3,2483,2509,0.625000,0.002010,0.004006,0.5028,0.998806,0.785323
8,9,5,4,2483,2508,0.555556,0.002010,0.004005,0.5026,0.998408,0.782393
9,10,6,4,2482,2508,0.600000,0.002412,0.004804,0.5028,0.998408,0.781070


In [9]:
final_prediction_result = fit_final_model_and_predict(
    X_train=X_stage2,
    y_train=y_train,
    X_test=X_test,
    selected_features=best_features,
    estimator_factory=tuned_factories.get(best_model_name, base_factories.get(best_model_name)),
    max_targets=1000,
)

# Filter predictions by the F1-selected threshold and rank by probability
predicted_probs = final_prediction_result.probabilities
threshold_mask = predicted_probs >= selected_threshold
above_threshold_indices = np.where(threshold_mask)[0]

# Sort by probability (descending)
sorted_indices = above_threshold_indices[np.argsort(-predicted_probs[above_threshold_indices])]
sorted_indices = sorted_indices[:1000]  # Limit to top 1000 if more are above threshold

final_prediction_frame = pd.DataFrame({
    "rank": np.arange(1, len(sorted_indices) + 1),
    "sample_index": sorted_indices,
    "probability": predicted_probs[sorted_indices],
})

final_prediction_frame.to_csv(OUTPUTS_PATH / "model_predictions.csv", index=False)

print(f"Samples above threshold: {len(sorted_indices)}")
final_prediction_frame.head()

Samples above threshold: 1000


,rank,sample_index,probability
0,1,927,0.799671
1,2,2161,0.799168
2,3,1089,0.796958
3,4,4738,0.792992
4,5,2251,0.789525
